# Stage 2 — GRPO Reinforcement Learning on PokerBench

Refines the SFT adapter using GRPO (Group Relative Policy Optimization).
For each scenario, the model generates 4 candidate actions which are scored
by the reward function — the model learns to prefer higher-scoring actions.

**Requires:** Stage 1 SFT adapter saved to Google Drive  
**Runtime:** GPU — A100 recommended  
**Expected time:** ~3-5 hrs for 5,000 steps  
**Output:** GRPO adapter saved to `MyDrive/pokerapp/grpo-adapter/`

### Before running
1. Runtime → Change runtime type → **A100 GPU**
2. Confirm `MyDrive/pokerapp/sft-adapter/` exists from Stage 1

## 1. Check GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found — change runtime to A100"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU : {gpu.name}")
print(f"VRAM: {gpu.total_memory / 1e9:.1f} GB")

## 2. Install dependencies

In [ ]:
# Install from GitHub to support the torch version on Kaggle
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q datasets "trl>=0.12.0" peft accelerate

## 3. Set directories

Detects whether running on Colab (uses Google Drive) or Kaggle (uses /kaggle/working/).

In [ ]:
import os

ON_KAGGLE = os.path.exists("/kaggle/working")

if ON_KAGGLE:
    # Path shown in the Kaggle sidebar when you add the model
    # Format: /kaggle/input/{username}/{model-name}/{framework}/{variation}/{version}
    # Check the exact path in the sidebar after adding your model
    KAGGLE_MODEL_PATH = "/kaggle/input/dominicvdb/pokerapp-sft-adapter/transformers/default/1"
    SFT_ADAPTER_DIR  = KAGGLE_MODEL_PATH
    GRPO_ADAPTER_DIR = "/kaggle/working/grpo-adapter"
    CHECKPOINT_DIR   = "/kaggle/working/grpo-checkpoints"
    print("Environment: Kaggle")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    SFT_ADAPTER_DIR  = "/content/drive/MyDrive/pokerapp/sft-adapter"
    GRPO_ADAPTER_DIR = "/content/drive/MyDrive/pokerapp/grpo-adapter"
    CHECKPOINT_DIR   = "/content/drive/MyDrive/pokerapp/grpo-checkpoints"
    print("Environment: Colab")

assert os.path.exists(SFT_ADAPTER_DIR), f"SFT adapter not found at {SFT_ADAPTER_DIR} — check KAGGLE_MODEL_PATH"
os.makedirs(GRPO_ADAPTER_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"SFT adapter      : {SFT_ADAPTER_DIR} ✓")
print(f"GRPO adapter dir : {GRPO_ADAPTER_DIR}")
print(f"Checkpoint dir   : {CHECKPOINT_DIR}")

## 4. Clone repo

In [ ]:
import sys, os, shutil

if ON_KAGGLE:
    # Copy uploaded src files into /kaggle/working/src/ so imports work as `from src.x import y`
    src_dir = "/kaggle/working/src"
    os.makedirs(src_dir, exist_ok=True)
    open(os.path.join(src_dir, "__init__.py"), "w").close()

    dataset_path = "/kaggle/input/datasets/dominicvdb/pokerapp-src"
    for f in os.listdir(dataset_path):
        if f.endswith(".py"):
            shutil.copy(os.path.join(dataset_path, f), os.path.join(src_dir, f))

    sys.path.insert(0, "/kaggle/working")
    print("src package ready:", os.listdir(src_dir))
else:
    REPO_DIR = "/content/pokerapp"
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        repo_url = f"https://{token}@github.com/dominicvdb/pokerapp.git"
    except Exception:
        repo_url = "https://github.com/dominicvdb/pokerapp.git"

    if os.path.exists(REPO_DIR):
        !cd {REPO_DIR} && git pull --quiet
        print("Repo updated")
    else:
        !git clone {repo_url} {REPO_DIR} --quiet
        print("Repo cloned")

    sys.path.insert(0, REPO_DIR)

## 5. Load dataset

In [ ]:
from src.data_loader import load_pokerbench

cache_dir = "/kaggle/working/data" if ON_KAGGLE else "/content/pokerapp/data"
dataset = load_pokerbench(cache_dir=cache_dir)
train_ds = dataset["train"]

print(f"Train rows: {len(train_ds):,}")
print(f"Test rows : {len(dataset['test']):,}")

## 6. Load Qwen3-8B + SFT adapter

Loads the base model with the Stage 1 adapter already applied.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print("SFT model loaded")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 7. Apply LoRA for GRPO

Smaller rank than SFT (r=16 vs r=32) to refine without overwriting what was learned.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing=False,
    random_state=42,
)

model.print_trainable_parameters()

## 8. Preprocess dataset into GRPO format

GRPO needs prompt-only inputs (no assistant message) and the ground truth answer
as a separate column so the reward function can score completions.

In [ ]:
from src.preprocessor import format_grpo, apply_chat_template

def preprocess_for_grpo(row):
    return {
        "prompt": apply_chat_template(
            format_grpo(row), tokenizer, add_generation_prompt=True
        ),
        "answer": row["output"],
    }

grpo_dataset = train_ds.map(
    preprocess_for_grpo,
    remove_columns=train_ds.column_names,
)

print(f"Processed {len(grpo_dataset):,} rows")
print(f"Columns: {grpo_dataset.column_names}")
print()
print("Sample prompt (truncated):")
print(grpo_dataset[0]["prompt"][:300], "...")
print(f"\nGround truth: {grpo_dataset[0]['answer']}")

## 9. Define reward function

GRPOTrainer calls this with a batch of completions and the matching ground truth
answers. Returns a score per completion that GRPO uses to rank candidates.

In [ ]:
from src.reward import poker_reward

def grpo_reward_fn(completions, answer=None, **kwargs):
    """Score each completion against its ground truth answer.

    Returns list of floats: 1.0 (correct), 0.5 (reasonable), 0.0 (bad sizing), -1.0 (wrong action).
    """
    return [poker_reward(completion, ans) for completion, ans in zip(completions, answer)]

# Sanity check
test_completions = ["bet 18", "fold", "call"]
test_answers     = ["bet 18", "fold", "raise 10"]
print("Reward function check:", grpo_reward_fn(test_completions, answer=test_answers))

## 10. Train with GRPOTrainer

For each prompt, the model generates `num_generations=4` candidate actions.
GRPO scores them, ranks them within the group, and updates weights to prefer better actions.

**Stability fixes vs first attempt:**
- `learning_rate=2e-5` (was 5e-5 — caused gradient explosion at step ~700)
- `max_grad_norm=0.1` — clips gradients before they explode
- `beta=0.1` — stronger KL penalty keeps model closer to reference (was 0.04)
- `num_generations=4` — more candidates per prompt = richer learning signal

**max_steps=2000** (~12 hrs on T4): set to 500 for a quick smoke test

In [ ]:
from trl import GRPOTrainer, GRPOConfig

MAX_STEPS = 2000

use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16
print(f"Precision: {'bf16' if use_bf16 else 'fp16'}  (GPU: {torch.cuda.get_device_name(0)})")

config = GRPOConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,        # reduced from 5e-5 to prevent gradient explosion
    beta=0.1,                  # stronger KL penalty (default 0.04 led to divergence)
    num_generations=4,
    max_completion_length=32,
    max_prompt_length=512,
    max_grad_norm=0.1,         # gradient clipping — critical for stability
    bf16=use_bf16,
    fp16=use_fp16,
    gradient_checkpointing=False,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_strategy="steps",
    save_steps=200,            # save frequently to recover from disconnections
    save_total_limit=2,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=grpo_reward_fn,
    args=config,
    train_dataset=grpo_dataset,
)

print(f"Max steps       : {MAX_STEPS:,}")
print(f"Effective batch : {2 * 2} prompts × {4} generations = {2 * 2 * 4} completions scored per update")

In [ ]:
import os

checkpoints = [
    d for d in os.listdir(CHECKPOINT_DIR)
    if d.startswith("checkpoint-")
] if os.path.exists(CHECKPOINT_DIR) else []

resume_from = CHECKPOINT_DIR if checkpoints else None
if resume_from:
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    print(f"Resuming from checkpoint: {latest}")
else:
    print("Starting from scratch")

trainer_stats = trainer.train(resume_from_checkpoint=resume_from)

print(f"\nTraining complete")
print(f"Runtime : {trainer_stats.metrics['train_runtime'] / 60:.1f} min")
print(f"Loss    : {trainer_stats.metrics['train_loss']:.4f}")

## 11. Save GRPO adapter

Saves to `GRPO_ADAPTER_DIR`:
- **Kaggle**: `/kaggle/working/grpo-adapter/` — download from the Output tab on the right
- **Colab**: `/content/drive/MyDrive/pokerapp/grpo-adapter/`

In [ ]:
model.save_pretrained(GRPO_ADAPTER_DIR)
tokenizer.save_pretrained(GRPO_ADAPTER_DIR)

print(f"GRPO adapter saved to {GRPO_ADAPTER_DIR}")
!ls -lh {GRPO_ADAPTER_DIR}

## 12. Spot check — compare SFT vs GRPO

Runs 20 test examples and scores with `poker_reward`.
Compare this result against the 75% from Stage 1 to confirm GRPO improved the model.

In [ ]:
import re
from src.reward import poker_reward

FastLanguageModel.for_inference(model)

_think_re = re.compile(r"<think>.*?</think>", re.DOTALL)

test_ds = dataset["test"].select(range(20))
correct = 0

for row in test_ds:
    prompt = apply_chat_template(
        format_grpo(row), tokenizer, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    generated_clean = _think_re.sub("", generated).strip()

    reward = poker_reward(generated_clean, row["output"])
    if reward == 1.0:
        correct += 1
    print(f"Expected: {row['output']:<12}  Predicted: {generated_clean:<12}  Reward: {reward}")

print(f"\nGRPO spot-check accuracy: {correct}/20 ({correct * 5}%)")
print(f"SFT baseline was 75% — improvement: {correct * 5 - 75:+}%")